# The `except ImportError: pass` Bug — and How to Fix It

**What this notebook teaches:** A recurring bug pattern in passagemath's modular codebase, how to diagnose it, and the endorsed fix using the `FeatureNotPresentError` system.

**Why it matters:** Several merged PRs (#2253, #2282, #2283) were built on this exact pattern. Once you understand it, you can find and fix these bugs independently.

**Structure:** Encounter → trace → fix → verify → exercise. Work in order.

In [ ]:
# Setup check — run this first.
# This notebook requires the 'passagemath' kernel (.venv-explore).
# If you get ModuleNotFoundError on sage imports, switch kernels:
#   Kernel > Change Kernel > passagemath

import sage.features
print("sage.features imported OK — you're on the right kernel.")

---
## Section 1: What is a modular install?

passagemath ships as roughly 100 separate pip packages. A user might install:

```
uv pip install passagemath-combinat
```

...but NOT `passagemath-flint`. That means `sage.libs.flint` is absent from their environment.

This is fine — as long as library code handles the missing dependency gracefully. The problem is when it doesn't.

The dangerous pattern looks like this at the top of a Python file:

```python
try:
    from some.optional.lib import some_function
except ImportError:
    pass
```

Looks reasonable — if the import fails, we skip it. But there's a hidden problem. Let's see it.

---
## Section 2: Encounter the bug

Run the cell below. It simulates exactly what happens when an optional library is absent.

In [ ]:
# Simulate: a fast counting library is not installed.
# The raise ImportError(...) line stands in for a real missing package.
try:
    raise ImportError("No module named 'fast_counter'")
    fast_count = lambda n: n * 42   # this line never runs
except ImportError:
    pass   # <-- what does this actually do to 'fast_count'?

def count_things(n):
    return fast_count(n)

count_things(5)   # run me — what happens?

You got `NameError: name 'fast_count' is not defined`.

**Why?**

- The `try` block raised `ImportError` before `fast_count = ...` ran
- The `except` block ran `pass` — which does exactly nothing
- So `fast_count` was **never assigned to anything**
- When `count_things(5)` tries to look it up, Python can't find it → `NameError`

`pass` doesn't "handle" the missing import. It silently leaves a name unbound — a time bomb that goes off later, far from where the real problem is.

---
## Section 3: Prove it — name binding

We can verify this directly:

In [ ]:
# Does 'x' exist in scope after a failed try block?
try:
    raise ImportError("missing")
    x = 42   # never runs
except ImportError:
    pass

print('x' in dir())   # False — x was never bound

The failure is **silent at import time** — you only discover it when you call the function that uses the missing name. That might be minutes later, in user code completely unrelated to the import.

---
## Section 4: The real code — `partition.py` before PR #2253

`src/sage/combinat/partition.py` provides `Partitions_n`, which counts integer partitions using FLINT for performance. Here's the relevant code **from git history, before the fix**:

**At the bottom of the file (module level):**
```python
try:
    from sage.libs.flint.arith_sage import number_of_partitions as flint_number_of_partitions
    cached_number_of_partitions = cached_function(flint_number_of_partitions)
except ImportError:
    pass   # BUG: cached_number_of_partitions is left unbound if flint is absent
```

**In class `Partitions_n`:**
```python
def cardinality(self, algorithm='flint'):
    if self.n < 0:
        return ZZ.zero()
    if algorithm == 'flint':
        return cached_number_of_partitions(self.n)   # NameError if flint is absent
    elif algorithm == 'gap':
        ...
```

**Plot twist: the bug is call-order dependent**

In an environment without FLINT, these two snippets behave completely differently:

```python
# Works fine:
p = Partitions(10)
p.list()        # category framework caches the list
p.cardinality() # returns 42 — FLINT path never touched
```

```python
# Crashes:
p = Partitions(10)
p.cardinality() # NameError: name 'cached_number_of_partitions' is not defined
```

Calling `.list()` first causes the category framework to cache the result and override `cardinality()` to return the length of that list. The broken FLINT path gets bypassed entirely. Call `.cardinality()` cold and it explodes.

Same install. Different call order. Different outcome. It only surfaces in the right environment with the right call sequence.

(Documented in the original issue, #2243.)

Same pattern as Section 2, now in production code. Try writing the fix yourself before reading Section 5 — you have everything you need:

1. Which line crashes, and only under what condition?
2. How should the `except ImportError` block be changed?
3. What should `cardinality()` do when FLINT isn't available?

Write your answer in the cell below, then continue.

In [ ]:
# Fix both marked lines, then run.

# Module level:
try:
    raise ImportError("pretend flint is absent")
    cached_number_of_partitions = lambda n: n * 42   # placeholder for the real function
except ImportError:
    pass   # FIX THIS LINE

# Usage site:
def cardinality(n):
    # FIX THIS FUNCTION — it should not raise NameError when flint is absent
    return cached_number_of_partitions(n)

# Goal: this should not raise NameError
cardinality(5)

---
## Section 5: The fix — three steps

Here is the pattern endorsed by mkoeppe (passagemath maintainer), from issue #2243.

Compare it to what you wrote, then go back and update your Section 4 cell to match.

**Step 1: Bind to `None` instead of `pass`**

```python
try:
    from sage.libs.flint.arith_sage import number_of_partitions as flint_number_of_partitions
    cached_number_of_partitions = cached_function(flint_number_of_partitions)
except ImportError:
    cached_number_of_partitions = None   # name is ALWAYS bound after this
```

The name is guaranteed to exist regardless of whether the import succeeded.

**Step 2: Check `is None` at the usage site**

**Step 3: Raise `FeatureNotPresentError` via the Feature system**

```python
def cardinality(self, algorithm='flint'):
    if self.n < 0:
        return ZZ.zero()
    if algorithm == 'flint':
        if cached_number_of_partitions is None:
            if self.n <= 10:
                return self._cardinality_from_iterator()   # slow enumeration fallback (defined in the class)
            from sage.features.sagemath import sage__libs__flint
            sage__libs__flint().require()   # raises FeatureNotPresentError with install hint
        return cached_number_of_partitions(self.n)
```

Two decisions baked into this:
- **Small-n fallback**: for `n <= 10`, enumerate directly — no FLINT needed. Basic doctests work without the optional dep.
- **Hard require for large n**: no reasonable fallback for `n = 10000`. Fail explicitly.

Not every fix needs a fallback. The `number_of_partitions()` standalone function (in the exercise) has no natural easy case — it calls `.require()` directly.

---
## Section 6: The Feature system

What is `sage__libs__flint()`?

`sage.features` is passagemath's registry for optional capabilities. Each `Feature` object knows:
- whether the capability is present in the current environment
- what package to install to get it

**Naming convention:** replace dots with double underscores.
- `sage.libs.flint` → `sage__libs__flint`
- `sage.libs.pari` → `sage__libs__pari`
- `sage.libs.ntl` → `sage__libs__ntl`

Let's see how it works:

In [ ]:
# Check whether FLINT is present in this environment
from sage.features.sagemath import sage__libs__flint

feature = sage__libs__flint()
print("Is FLINT present?", feature.is_present())

In [ ]:
# Now see what .require() does when a feature is absent.
# NTL (Number Theory Library) is not installed in this environment.
from sage.features.sagemath import sage__libs__ntl

ntl = sage__libs__ntl()
print("Is NTL present?", ntl.is_present())   # should print False
print()

# Now call .require() — this is what the fixed code does at the call site
try:
    ntl.require()
except Exception as e:
    print(type(e).__name__)
    print(e)

The error message tells the user *what is missing* and *what to install*.

| Situation | Error type | Actionable? |
|---|---|---|
| `except ImportError: pass` | `NameError: name 'cached_number_of_partitions' is not defined` | No — looks like a library bug |
| `= None` + `.require()` | `FeatureNotPresentError` (see cell output above) | Yes — tells you exactly what to install |

This is what makes passagemath usable as a modular library: failures point to the missing package, not to internals.

---
## Section 7: Verify the fix end-to-end

Here's the before and after, both runnable. Compare the output.

In [ ]:
# BEFORE (the bug) — self-contained
try:
    raise ImportError("No module named 'fast_counter'")
    _fast_count_buggy = lambda n: n * 42
except ImportError:
    pass   # _fast_count_buggy never bound

def count_things_buggy(n):
    return _fast_count_buggy(n)

try:
    count_things_buggy(5)
except NameError as e:
    print("BUG —", type(e).__name__ + ":", e)

In [ ]:
# AFTER (the fix) — self-contained
try:
    raise ImportError("No module named 'fast_counter'")
    _fast_count_fixed = lambda n: n * 42
except ImportError:
    _fast_count_fixed = None   # always bound

def count_things_fixed(n):
    if _fast_count_fixed is None:
        raise RuntimeError(
            "fast_counter is not installed.\n"
            "To install: uv pip install passagemath-fast-counter"
        )
    return _fast_count_fixed(n)

try:
    count_things_fixed(5)
except RuntimeError as e:
    print("FIXED —", type(e).__name__ + ":", e)

The pattern holds. Now apply it to real code.

---
## Section 8: Exercise

Here is `number_of_partitions()` — a standalone function in `partition.py` with the **same bug** at a different call site. This is from the actual pre-fix git state.

```python
# number_of_partitions() in partition.py — BEFORE the fix
def number_of_partitions(n, algorithm='default'):
    r"""
    Return the number of partitions of `n`.
    """
    n = ZZ(n)
    if n < 0:
        raise ValueError("n must be a non-negative integer")

    algorithm = 'flint'   # (simplified from original)

    if algorithm == 'flint':
        return cached_number_of_partitions(n)   # NameError if flint absent

    raise ValueError("unknown algorithm '%s'" % algorithm)
```

**Your task:** write the fixed version in the cell below.

A few things to keep in mind:
- `cached_number_of_partitions` is the same module-level name (already `= None` from the module-level fix)
- Use `sage__libs__flint().require()` — import it from `sage.features.sagemath`
- Should this function have a small-n fallback like `cardinality()` did? Why or why not?

In [ ]:
# YOUR ANSWER HERE
# Step 1 is already done: cached_number_of_partitions is bound to None (flint is absent).
# Your task: add the usage-site guard so this raises FeatureNotPresentError, not TypeError.

cached_number_of_partitions = None   # simulating: flint is absent

def number_of_partitions(n):
    if n < 0:
        raise ValueError("n must be a non-negative integer")
    # YOUR FIX HERE
    # from sage.features.sagemath import sage__libs__flint
    # ...
    return cached_number_of_partitions(n)   # currently raises TypeError: 'NoneType' object is not callable

number_of_partitions(100)

Once your cell raises a `FeatureNotPresentError` instead of `TypeError`, you've got it right. Then compare with the answer below.

---
### Answer — from the actual PR #2253 diff

```python
if algorithm == 'flint':
    if cached_number_of_partitions is None:
        from sage.features.sagemath import sage__libs__flint
        sage__libs__flint().require()
    return cached_number_of_partitions(n)
```

No small-n fallback. This function is a pure lookup — there's no "easy case". Better to fail immediately with a clear message than to silently enumerate for small inputs and crash for large ones.

The full PR diff is 9 lines of logic across 3 call sites + 1 line changing `pass` → `= None`.

**PR:** https://github.com/passagemath/passagemath/pull/2253

---
## Summary

The complete pattern:

```python
# Module level
try:
    from some.optional.lib import some_func
except ImportError:
    some_func = None               # NOT pass

# Usage site
def my_method(self):
    if some_func is None:
        if self.n <= THRESHOLD:    # optional: graceful fallback for easy cases
            return self._slow_fallback()
        from sage.features.sagemath import sage__libs__something
        sage__libs__something().require()   # explicit error with install hint
    return some_func(self.n)
```

**To find instances in the passagemath codebase:**

```bash
python tools/check_unbound_imports.py src/sage/
```

This AST checker (added in PR #2283) finds names that are conditionally assigned in a `try` block and used later without a guard.